In [1]:
import os
import sys
import shutil
import subprocess

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [2]:
utils_path = !pwd
utils_path = utils_path[0]
utils_path = os.path.join(utils_path, '..', 'src', 'utils')
sys.path.append(utils_path)
utils_path

'/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/notebooks/../src/utils'

In [3]:
from InventoryTransformations import InventoryTransformations

In [4]:
# os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/BIN:{os.environ.get('PATH', '')}"
# os.environ["PATH"]

In [5]:
# os.environ.get("JAVA_HOME")
# shutil.which("java")
# subprocess.run(["java", "-version"], capture_output=True, text=True).stderr

In [6]:
JARS_URLS = ",".join([
    "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar",
    "https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar",
])

In [7]:
DRIVER_HOST = "host.docker.internal"
DRIVER_PORT = "4042"
BLOCK_MANAGER_PORT = "4043"

In [8]:
# TODO: MAKE SURE TO RUN THIS IN ORDER TO HANDLE host.docker.internal in local host:
#  echo "127.0.0.1 host.docker.internal" | sudo tee -a /etc/hosts 

AWS_BUNDLE = "com.amazonaws:aws-java-sdk-bundle:1.12.262"
HADOOP_AWS = "org.apache.hadoop:hadoop-aws:3.3.4"

spark = (
    SparkSession.builder
    .appName("data_cleansing")
    .master("spark://localhost:7077")          # driver the notebook
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.host", DRIVER_HOST)
    .config("spark.driver.port", DRIVER_PORT)
    .config("spark.blockManager.port", BLOCK_MANAGER_PORT)
    # Including dependencies
    .config("spark.submit.pyFiles", f"{utils_path}/InventoryTransformations.py")
    # ↓ Let Spark fetch & ship jars to executors
    .config("spark.jars.packages", f"{HADOOP_AWS},{AWS_BUNDLE}")
    # MINIO/S3A
    .config("spark.hadoop.fs.s3a.endpoint", "http://host.docker.internal:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    # Explicit creds provider (so the access/secret keys below are actually used)
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    # Failure settings
    .config("spark.hadoop.fs.s3a.connection.timeout", "10000")
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000")
    .config("spark.hadoop.fs.s3a.socket.timeout", "30000")
    .config("spark.hadoop.fs.s3a.attempts.maximum", "3")
    .config("spark.hadoop.fs.s3a.retry.limit", "2")
    .getOrCreate()
)

spark.version, spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()

25/11/09 20:51:44 WARN Utils: Your hostname, Jonathans-MacBook-6.local resolves to a loopback address: 127.0.0.1; using 172.16.194.67 instead (on interface en0)
25/11/09 20:51:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/jonathanperalgort/.ivy2/cache
The jars for the packages stored in: /Users/jonathanperalgort/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-72ec916c-f6f1-4b74-abf3-c1669c0eddfa;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 466ms :: artifacts dl 15ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | nu

('3.4.0', '3.3.4')

In [9]:
hadoop_config = spark._jsc.hadoopConfiguration()
print(hadoop_config.get("fs.s3a.endpoint"))
print(hadoop_config.get("fs.s3a.aws.credentials.provider"))
print(hadoop_config.get("fs.s3a.path.style.access"))
print(hadoop_config.get("fs.s3a.connection.ssl.enabled"))

http://host.docker.internal:9000
org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider
true
false


In [10]:
jvm = spark.sparkContext._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI("s3a://inventory/"), hadoop_config)
print(fs)
for s in fs.listStatus(jvm.org.apache.hadoop.fs.Path("s3a://inventory/")):
    print("->", s.getPath().toString())

25/11/09 20:51:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


S3AFileSystem{uri=s3a://inventory, workingDir=s3a://inventory/user/jonathanperalgort, inputPolicy=normal, partSize=67108864, enableMultiObjectsDelete=true, maxKeys=5000, readAhead=65536, blockSize=33554432, multiPartThreshold=134217728, s3EncryptionAlgorithm='NONE', blockFactory=org.apache.hadoop.fs.s3a.S3ADataBlocks$DiskBlockFactory@12f3ebc8, auditManager=Service NoopAuditManagerS3A in state NoopAuditManagerS3A: STARTED, metastore=NullMetadataStore, authoritativeStore=false, authoritativePath=[], useListV1=false, magicCommitter=true, boundedExecutor=BlockingThreadPoolExecutorService{SemaphoredDelegatingExecutor{permitCount=160, available=160, waiting=0}, activeCount=0}, unboundedExecutor=java.util.concurrent.ThreadPoolExecutor@340cc941[Running, pool size = 0, active threads = 0, queued tasks = 0, completed tasks = 0], credentials=AWSCredentialProviderList[refcount= 1: [SimpleAWSCredentialsProvider], delegation tokens=disabled, DirectoryMarkerRetention{policy='delete'}, instrumentation

In [11]:
spark._jsc.hadoopConfiguration().get("fs.s3a.impl")

'org.apache.hadoop.fs.s3a.S3AFileSystem'

In [12]:
spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()

# Driver JVM classpath (you'll see site-packages/pyspark/jars but no hadoop-aws/aws-sdk)
spark.sparkContext._jvm.java.lang.System.getProperty("java.class.path")

'/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.10/site-packages/pyspark/conf:/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.10/site-packages/pyspark/jars/dropwizard-metrics-hadoop-metrics2-reporter-0.1.2.jar:/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.10/site-packages/pyspark/jars/netty-handler-proxy-4.1.87.Final.jar:/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.10/site-packages/pyspark/jars/logging-interceptor-3.12.12.jar:/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.10/site-packages/pyspark/jars/threeten-extra-1.7.1.jar:/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.10/site-packages/pyspark/jars/hive-shims-2.3.9.jar:/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/venv/lib/

In [13]:
# TODO: REPLACE LOCAL CSV LOADING WITH A DISTRIBUTED STORAGE SOLUTION (E.G. S3)
df_raw = spark.read.csv("s3a://inventory/inventory_raw.csv", header=True, inferSchema=True)

In [14]:
df_raw.show()

+-------------+---------------+----------+-------------+-----------------+--------------------+-----------+-------------+--------------------+
|source_row_id|             ip|  hostname|         fqdn|              mac|               owner|device_type|         site|               notes|
+-------------+---------------+----------+-------------+-----------------+--------------------+-----------+-------------+--------------------+
|            1|192.168.010.005|    HOST01|         null|AA-BB-CC-DD-EE-FF|priya (platform) ...|     server|   BLR Campus|             db host|
|            2|     10.0.1.300|   host-02|host-02.local|11-22-33-44-55-66|                 ops|       null|    HQ Bldg 1|            edge gw?|
|            3|         10.0.1|    host03|         null|   aabb.ccdd.eeff|jane@corp.example...|     switch|HQ-BUILDING-1|                null|
|            4|     10.0.1.1.2|printer-01|         null|00:11:22:33:44:55|          Facilities|    printer|           HQ|                null|

### SETTING Inventory_Transformations' FUNCTIONS RETURN STRUCTS

In [15]:
traceability_schema = T.StructType([
    T.StructField("field", T.StringType(), True),
    T.StructField("from", T.StringType(), True),
    T.StructField("to", T.StringType(), True),
    T.StructField("reason", T.StringType(), True)
])

In [16]:
ipv4_schema = T.StructType([
    T.StructField("ip_valid", T.BooleanType(), True),
    T.StructField("ip_canonical", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True)
])

In [17]:
hostname_schema = T.StructType([
    T.StructField("hostname_valid", T.BooleanType(), True), 
    T.StructField("hostname_canonical", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True)
])

In [18]:
site_schema = T.StructType([
    T.StructField("site_normalized", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True)
])

In [19]:
fqdn_schema = T.StructType([
    T.StructField("fqdn_valid", T.BooleanType(), True),
    T.StructField("fqdn_canonical", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True),
    T.StructField("fqdn_consistent", T.StringType(), True)
])

In [20]:
mac_schema = T.StructType([
    T.StructField("mac_valid", T.BooleanType(), True),
    T.StructField("mac_canonical", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True)
])

In [21]:
owner_schema = T.StructType([
    T.StructField("owner", T.StringType(), True),
    T.StructField("owner_email", T.StringType(), True),
    T.StructField("owner_team", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True)
])

In [22]:
device_type_schema = T.StructType([
    T.StructField("device_type", T.StringType(), True),
    T.StructField("device_type_confidence", T.IntegerType(), True),
    T.StructField("tr_metadata", traceability_schema, True),
])

### UDFs

In [23]:
it = InventoryTransformations()

IP

In [24]:
ipv4_udf = F.udf(lambda x: it.ipv4_validate_and_normalize(x), ipv4_schema)

In [25]:
ipv4_type_udf = F.udf(lambda x: it.classify_ipv4_type(x), T.StringType())

In [26]:
default_subnet_udf = F.udf(lambda ip, ip_type: it.default_subnet(ip, ip_type), T.StringType())

HOSTNAME

In [27]:
hostname_udf = F.udf(lambda x: it.validate_hostname(x), hostname_schema)

SITE

In [28]:
site_udf = F.udf(lambda x: it.normalize_site(x), site_schema)

In [29]:
reverse_ptr_udf = F.udf(lambda x: it.generate_reverse_ptr(x), T.StringType())

FQDN

In [59]:
fqdn_udf = F.udf(lambda fqdn, hostname, site: it.validate_fqdn(fqdn, hostname, site), fqdn_schema)

MAC

In [31]:
mac_udf = F.udf(lambda x: it.validate_mac(x), mac_schema)

OWNER

In [32]:
owner_udf = F.udf(lambda x: it.parse_owner(x), owner_schema)

DEVICE_TYPE

In [33]:
device_type_udf = F.udf(lambda x: it.normalize_device_type(x), device_type_schema)

### TRANSFORMING INVENTORY

In [39]:
df = df_raw

In [40]:
df.show()

+-------------+---------------+----------+-------------+-----------------+--------------------+-----------+-------------+--------------------+
|source_row_id|             ip|  hostname|         fqdn|              mac|               owner|device_type|         site|               notes|
+-------------+---------------+----------+-------------+-----------------+--------------------+-----------+-------------+--------------------+
|            1|192.168.010.005|    HOST01|         null|AA-BB-CC-DD-EE-FF|priya (platform) ...|     server|   BLR Campus|             db host|
|            2|     10.0.1.300|   host-02|host-02.local|11-22-33-44-55-66|                 ops|       null|    HQ Bldg 1|            edge gw?|
|            3|         10.0.1|    host03|         null|   aabb.ccdd.eeff|jane@corp.example...|     switch|HQ-BUILDING-1|                null|
|            4|     10.0.1.1.2|printer-01|         null|00:11:22:33:44:55|          Facilities|    printer|           HQ|                null|

IPV4 TRANSFORMATIONS

In [46]:
df = (df.withColumn("ipv4", ipv4_udf(F.col("ip")))
      .withColumn("ip_valid", F.col("ipv4.ip_valid"))
      .withColumn("ip_canonical", F.col("ipv4.ip_canonical"))
      .withColumn("ip_tr_metadata", F.col("ipv4.tr_metadata")))

+----------------------------------------------------+
|ip_tr_metadata                                      |
+----------------------------------------------------+
|{ip, 192.168.010.005, 192.168.10.5, ok}             |
|{ip, 10.0.1.300, null, octet_out_of_range}          |
|{ip, 10.0.1, null, wrong_octet_count}               |
|{ip, 10.0.1.1.2, null, wrong_octet_count}           |
|{ip, fe80::1%eth0, null, ipv6_or_non_ipv4}          |
|{ip, 127.0.0.1, 127.0.0.1, ok}                      |
|{ip, 169.254.10.20, 169.254.10.20, ok}              |
|{ip,   10.10.10.10  , 10.10.10.10, ok}              |
|{ip, abc.def.ghi.jkl, null, non_numeric_or_negative}|
|{ip, 192.168.1.-1, null, non_numeric_or_negative}   |
|{ip, 192.168.1.255, 192.168.1.255, ok}              |
|{ip, 192.168.1.0, 192.168.1.0, ok}                  |
|{ip, 8.8.8.8, 8.8.8.8, ok}                          |
|{ip, 010.010.010.010, 10.10.10.10, ok}              |
|{ip, N/A, null, wrong_octet_count}                  |
+---------

In [ ]:
df = df.withColumn("ip_type", ipv4_type_udf(F.col("ip_canonical")))

+----------------+
|ip_type         |
+----------------+
|private_rfc1918 |
|invalid         |
|invalid         |
|invalid         |
|invalid         |
|loopback        |
|link_local_apipa|
|private_rfc1918 |
|invalid         |
|invalid         |
|private_rfc1918 |
|private_rfc1918 |
|public_or_other |
|private_rfc1918 |
|invalid         |
+----------------+



In [ ]:
df = df.withColumn("subnet_cidr", F.when(F.col("ip_valid") == True, default_subnet_udf(F.col('ip_canonical'), F.col("ip_type"))).otherwise(F.lit(None)))


+---------------+
|subnet_cidr    |
+---------------+
|192.168.10.0/24|
|null           |
|null           |
|null           |
|null           |
|127.0.0.0/8    |
|169.254.0.0/16 |
|10.0.0.0/8     |
|null           |
|null           |
|192.168.1.0/24 |
|192.168.1.0/24 |
|8.8.8.0/24     |
|10.0.0.0/8     |
|null           |
+---------------+



HOSTNAME TRANSFORMATIONS

In [ ]:
df = (
    df.withColumn("hr", hostname_udf(F.col("hostname")))
        .withColumn("hostname_valid", F.col("hr.hostname_valid"))
        .withColumn("hostname_canonical", F.col("hr.hostname_canonical"))
        .withColumn("hostname_tr_metadata", F.col("hr.tr_metadata"))
        .drop("hr")
)

+--------------+------------------+--------------------------------------+
|hostname_valid|hostname_canonical|hostname_tr_metadata                  |
+--------------+------------------+--------------------------------------+
|true          |host01            |{hostname, HOST01, host01, ok}        |
|true          |host-02           |{hostname, host-02, host-02, ok}      |
|true          |host03            |{hostname, host03, host03, ok}        |
|true          |printer-01        |{hostname, printer-01, printer-01, ok}|
|true          |iot-cam01         |{hostname, iot-cam01, iot-cam01, ok}  |
|true          |local-test        |{hostname, local-test, local-test, ok}|
|true          |host-apipa        |{hostname, host-apipa, host-apipa, ok}|
|true          |srv-10            |{hostname, srv-10, srv-10, ok}        |
|true          |badhost           |{hostname, badhost, badhost, ok}      |
|true          |neg               |{hostname, neg, neg, ok}              |
|true          |bcast    

SITE TRANSFORMATIONS

In [ ]:
df = (
    df.withColumn("site_res", site_udf(F.col("site")))
        .withColumn("site_normalized", F.col("site_res.site_normalized"))
        .withColumn("site_tr_metadata", F.col("site_res.tr_metadata"))
        .drop("site_res")
)


+---------------+-------------------------------------------------+
|site_normalized|site_tr_metadata                                 |
+---------------+-------------------------------------------------+
|blr-campus     |{site, BLR Campus, blr-campus, normalized_site}  |
|hq-bldg-1      |{site, HQ Bldg 1, hq-bldg-1, normalized_site}    |
|hq-bldg-1      |{site, HQ-BUILDING-1, hq-bldg-1, normalized_site}|
|hq             |{site, HQ, hq, normalized_site}                  |
|lab-1          |{site, Lab-1, lab-1, normalized_site}            |
|unknown        |{site, N/A, unknown, missing}                    |
|unknown        |{site, null, unknown, missing}                   |
|blr-campus     |{site, BLR campus, blr-campus, normalized_site}  |
|unknown        |{site, null, unknown, missing}                   |
|unknown        |{site, null, unknown, missing}                   |
|unknown        |{site, null, unknown, missing}                   |
|unknown        |{site, null, unknown, missing} 

FQDN TRANSFORMATIONS

In [ ]:
df = df.withColumn("reverse_ptr", reverse_ptr_udf(F.col("ip_canonical")))

df = (
    df.withColumn("fqdn_res", fqdn_udf(F.col('fqdn'), F.col("hostname_canonical"), F.col("site_normalized")))
        .withColumn("fqdn_valid", F.col("fqdn_res.fqdn_valid"))
        .withColumn("fqdn_canonical", F.col("fqdn_res.fqdn_canonical"))
        .withColumn("fqdn_tr_metadata", F.col("fqdn_res.tr_metadata"))
        .withColumn("fqdn_consistent", F.col("fqdn_res.fqdn_consistent"))
        .drop("fqdn_res")
)

+----------+--------------+-------------------------------------------+---------------+
|fqdn_valid|fqdn_canonical|fqdn_tr_metadata                           |fqdn_consistent|
+----------+--------------+-------------------------------------------+---------------+
|false     |null          |{fqdn, null, null, missing}                |inconsistent   |
|true      |host-02.local |{fqdn, host-02.local, host-02.local, valid}|inconsistent   |
|false     |null          |{fqdn, null, null, missing}                |inconsistent   |
|false     |null          |{fqdn, null, null, missing}                |inconsistent   |
|false     |null          |{fqdn, null, null, missing}                |inconsistent   |
|false     |null          |{fqdn, null, null, missing}                |inconsistent   |
|false     |null          |{fqdn, null, null, missing}                |inconsistent   |
|false     |null          |{fqdn, null, null, missing}                |inconsistent   |
|false     |null          |{fqdn

MAC TRANSFORMATIONS

In [ ]:
df = (
    df.withColumn("mac_res", mac_udf(F.col("mac")))
        .withColumn("mac_valid", F.col("mac_res.mac_valid"))
        .withColumn("mac_canonical", F.col("mac_res.mac_canonical"))
        .withColumn("mac_tr_metadata", F.col("mac_res.tr_metadata"))
        .drop("mac_res")
)


+---------+-----------------+--------------------------------------------------+
|mac_valid|mac_canonical    |mac_tr_metadata                                   |
+---------+-----------------+--------------------------------------------------+
|true     |aa:bb:cc:dd:ee:ff|{mac, AA-BB-CC-DD-EE-FF, aa:bb:cc:dd:ee:ff, valid}|
|true     |11:22:33:44:55:66|{mac, 11-22-33-44-55-66, 11:22:33:44:55:66, valid}|
|true     |aa:bb:cc:dd:ee:ff|{mac, aabb.ccdd.eeff, aa:bb:cc:dd:ee:ff, valid}   |
|true     |00:11:22:33:44:55|{mac, 00:11:22:33:44:55, 00:11:22:33:44:55, valid}|
|true     |00:aa:bb:cc:dd:ee|{mac, 00:aa:bb:cc:dd:ee, 00:aa:bb:cc:dd:ee, valid}|
|false    |null             |{mac, null, null, missing}                        |
|false    |null             |{mac, null, null, missing}                        |
|false    |null             |{mac, null, null, missing}                        |
|false    |null             |{mac, null, null, missing}                        |
|false    |null             

OWNER PARSING

In [ ]:
df = (
    df.withColumn("owner_res", owner_udf(F.col("owner")))
        .withColumn("owner_normalized", F.col("owner_res.owner"))
        .withColumn("owner_email", F.col("owner_res.owner_email"))
        .withColumn("owner_team", F.col('owner_res.owner_team'))
        .withColumn("owner_tr_metadata", F.col("owner_res.tr_metadata"))
        .drop("owner_res")
)


+----------------+----------------------+----------+-----------------------------------------------------------------------------------+
|owner_normalized|owner_email           |owner_team|owner_tr_metadata                                                                  |
+----------------+----------------------+----------+-----------------------------------------------------------------------------------+
|priya           |priya@corp.example.com|platform  |{owner, priya (platform) priya@corp.example.com, priya, owner_valid_and_normalized}|
|ops             |null                  |null      |{owner, ops, ops, owner_valid_and_normalized}                                      |
|null            |jane@corp.example.com |null      |{owner, jane@corp.example.com, null, owner_valid_and_normalized}                   |
|facilities      |null                  |null      |{owner, Facilities, facilities, owner_valid_and_normalized}                        |
|sec             |null                  |

DEVICE TYPE TRANSFORMATIONS

In [ ]:
df = (
    df.withColumn("dt_res", device_type_udf(F.col("device_type")))
        .withColumn("device_type_normalized", F.col("dt_res.device_type"))
        .withColumn("device_type_confidence", F.col("dt_res.device_type_confidence"))
        .withColumn("device_type_tr_metadata", F.col("dt_res.tr_metadata"))
        .drop("dt_res")
)


+----------------------+----------------------+--------------------------------------------+
|device_type_normalized|device_type_confidence|device_type_tr_metadata                     |
+----------------------+----------------------+--------------------------------------------+
|server                |100                   |{device_type, server, server, exact_match}  |
|unknown               |0                     |{device_type, null, unknown, missing}       |
|switch                |100                   |{device_type, switch, switch, exact_match}  |
|printer               |100                   |{device_type, printer, printer, exact_match}|
|unknown               |0                     |{device_type, iot, storage, no_match}       |
|unknown               |0                     |{device_type, null, unknown, missing}       |
|unknown               |0                     |{device_type, null, unknown, missing}       |
|server                |100                   |{device_type, server, s

NORMALIZATION STEPS

In [67]:
df = df.withColumn(
    "normalization_steps",
    F.array(
        "ip_tr_metadata",
        "hostname_tr_metadata",
        "fqdn_tr_metadata",
        "mac_tr_metadata",
        "owner_tr_metadata",
        "device_type_tr_metadata",
        "site_tr_metadata"
    )
)
df.select("normalization_steps").show(truncate=False)


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|normalization_steps                                                                                                                                                                                                                                                                                                                         |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

DF_FINAL

In [ ]:
# TODO:WRITE DF_FINAL